# NovaChat - Fine-Tune on Colab (GPU)

Train the NovaChat LoRA adapter in a few minutes on a free GPU.

**Steps:**
1. Run the cells below in order (Runtime > Run all)
2. When asked, upload `data/training_data.json` from your NovaChat project
3. After training, download `nova_lora.zip` and extract it over your local `models/lora/` folder

Base model: `HuggingFaceTB/SmolLM2-1.7B-Instruct`

In [ ]:
%pip install -q transformers peft datasets accelerate
%pip uninstall -q -y torchao
print("Setup complete")

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

In [ ]:
import os
from google.colab import files

DATA_PATH = "/content/training_data.json"

if os.path.exists(DATA_PATH):
    print("training_data.json already present.")
else:
    print("Upload training_data.json from your NovaChat/data folder:")
    uploaded = files.upload()
    for name in uploaded:
        with open(DATA_PATH, "wb") as f:
            f.write(uploaded[name])
        print(f"Saved as {DATA_PATH}")

In [ ]:
import json

import torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)

MODEL_NAME = "HuggingFaceTB/SmolLM2-1.7B-Instruct"
DATA_PATH = "/content/training_data.json"
OUTPUT_DIR = "/content/models/lora"
EPOCHS = 3
MAX_LENGTH = 512


tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float16)
model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

with open(DATA_PATH, encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data)} training examples")


def tokenize(example):
    prefix = f"### Instruction:\n{example['instruction']}\n\n### Response:\n"
    prefix_ids = tokenizer(prefix, add_special_tokens=False)["input_ids"]
    response_ids = tokenizer(f"{example['response']}\n", add_special_tokens=False)["input_ids"]
    if not response_ids:
        response_ids = [tokenizer.eos_token_id]
    combined = (prefix_ids + response_ids)[:MAX_LENGTH]
    combined = combined + [tokenizer.pad_token_id] * (MAX_LENGTH - len(combined))
    labels = ([-100] * len(prefix_ids) + response_ids)[:MAX_LENGTH]
    labels = labels + [-100] * (len(combined) - len(labels))
    return {
        "input_ids": combined,
        "labels": labels,
        "attention_mask": [1 if t != tokenizer.pad_token_id else 0 for t in combined],
    }


dataset = Dataset.from_list(raw_data)
tokenized_dataset = dataset.map(tokenize, remove_columns=dataset.column_names)
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer, padding="max_length", max_length=MAX_LENGTH
)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=2e-4,
    logging_steps=5,
    save_strategy="epoch",
    fp16=True,
    report_to=[],
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

trainer.train()

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"LoRA adapter saved to {OUTPUT_DIR}")


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/nova_lora", "zip", "/content/models/lora")
files.download("/content/nova_lora.zip")
print("Extract this zip over your local NovaChat/models/lora/ folder, then run uvicorn backend.app:app --reload")